# Monte Carlo: BLER, Confidence Intervals, and Threads

Monte Carlo estimates the decoder's error probability over many random transmissions. This notebook counts exact frame errors, builds 95% Wilson intervals, and measures batch throughput. A fixed seed makes the example reproducible.

The interval represents uncertainty caused by the finite number of simulations. For an engineering estimate, increase the number of frames until the interval is sufficiently narrow.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "Cargo.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "Cargo.toml").exists():
    raise RuntimeError("Open this notebook from inside the ra-rust project")
sys.path.insert(0, str(ROOT / "python"))
from ra_reconcile import decode, decode_batch, encode
MATRIX_SMALL = ROOT / "data" / "H_small.edges"
MATRIX_RA = ROOT / "data" / "RA_n6000_R150.edges"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter

SEED = 731204
P = 0.01
FRAMES = 64
THREAD_COUNTS = [1, 2, 4]
MAX_ITER = 100
rng = np.random.default_rng(SEED)
k = 120
information = "".join(rng.integers(0, 2, size=k).astype(str))
codeword = encode(MATRIX_RA, information)
c = np.fromiter((int(bit) for bit in codeword), dtype=np.uint8)
bobs = rng.integers(0, 2, size=(FRAMES, len(c)), dtype=np.uint8)
errors = (rng.random((FRAMES, len(c))) < P).astype(np.uint8)
alice = bobs ^ errors
public = bobs ^ c
a_frames = ["".join(row.astype(str)) for row in alice]
m_frames = ["".join(row.astype(str)) for row in public]
print(f"Seed={SEED}; p={P}; frames={FRAMES}; n={len(c)}")

## Frame Errors and the Wilson Interval

A frame is an error if the estimated codeword differs from the transmitted word, even when the syndrome is zero. For `x` errors in `N` trials, the observed rate is `x/N`. The Wilson interval is more reliable than a simple normal interval, especially when there are few errors.


In [ ]:
def wilson_interval(errors, trials, z=1.96):
    phat = errors / trials
    den = 1 + z*z/trials
    center = (phat + z*z/(2*trials)) / den
    half = z*np.sqrt(phat*(1-phat)/trials + z*z/(4*trials*trials)) / den
    return max(0.0, center-half), min(1.0, center+half)

rows = []
for workers in THREAD_COUNTS:
    started = perf_counter()
    results = decode_batch(MATRIX_RA, a_frames, m_frames, P,
                           max_iter=MAX_ITER, threads=workers)
    wall_ms = (perf_counter() - started) * 1000
    failures = sum(r.estimate != codeword for r in results)
    lo, hi = wilson_interval(failures, FRAMES)
    rows.append({"threads": workers, "frames": FRAMES, "exact_frame_errors": failures,
                 "bler": failures/FRAMES, "bler_ci_low": lo, "bler_ci_high": hi,
                 "syndrome_failures": sum(not r.success for r in results),
                 "wall_ms": wall_ms, "frames_per_second": FRAMES/(wall_ms/1000)})
mc = pd.DataFrame(rows)
mc

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mid = mc.bler.to_numpy()
axes[0].errorbar(mc.threads, mid,
                 yerr=[mid-mc.bler_ci_low, mc.bler_ci_high-mid],
                 fmt="o", capsize=5)
axes[0].set(xlabel="Workers across frames", ylabel="BLER (95% CI)", ylim=(-.03, 1.03))
axes[0].grid(alpha=.3)
axes[1].bar(mc.threads.astype(str), mc.frames_per_second, color="steelblue")
axes[1].set(xlabel="Workers across frames", ylabel="Frames per second")
axes[1].grid(axis="y", alpha=.3)
fig.suptitle(f"RA-6000 Monte Carlo: p={P}, N={FRAMES}")
fig.tight_layout()
plt.show()

## Interpreting the Results

The same fixed set of frames is used for every worker count, so the error rate should remain the same; only runtime changes. With few frames, a single rare failure can strongly affect the BLER estimate, resulting in a wide confidence interval. Increasing `FRAMES` improves statistical precision. Runtime depends on the processor, matrix, and iteration count; more workers are not always faster because of overhead and memory bandwidth limits.
